# Oracle Fleet Miner (Colab L4 / A100)

**Goal:** mine 10,000+ crisis-state anchors with K=32 common-RNG rollouts per move at top-6 candidates, H=300 horizon. Output is a `phase1_oracle.pt`-compatible file for Path B (fresh policy training with rollout-oracle corrections).

**Architecture:** single-process CUDA, **fleet of M=512 concurrent rollouts** advanced in lockstep. Each step: build M observations, ONE GPU forward at bs=M, advance all games. Dead games auto-refill from the work queue. Much faster than the M5 MAX multiprocessing design (bs=16 per server batch).

**Expected runtime on L4** (10K crisis anchors, K=32, top-6, H=300, fleet=512):
- Phase A (top-K per anchor): ~30 s
- Phase B (afterstates): ~30 s
- Phase D (fleet rollout): **~2-4 h**
- Phase E (aggregate + save): ~10 s

vs M5 MAX 16-worker: ~16 h for the same workload.

## Upload to `MyDrive/alphatrain/` before running:
1. `colorlines_fleet_mine.tar.gz` — built locally by `scripts/build_fleet_tarball.sh`. ~7-8 GB (code + crisis_v12 + selfplay_v12 + model).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/alphatrain'

# Copy + extract tarball
!cp {DRIVE}/colorlines_fleet_mine.tar.gz /content/
!cd /content && tar xzf colorlines_fleet_mine.tar.gz

# Quick contents check
!ls -la /content/alphatrain/data/pillar2z_epoch_19.pt
!ls /content/data/crisis_v12 | head -3 && echo '...'
!ls /content/data/crisis_v12 | wc -l
print('--- selfplay ---')
!ls /content/data/selfplay_v12 | head -3 && echo '...'
!ls /content/data/selfplay_v12 | wc -l

In [ ]:
# Install deps. Colab has torch + numpy preinstalled; numba may need install.
!pip install -q numba

In [ ]:
# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')
    print(f'SM count: {g.multi_processor_count}')
# Expected: NVIDIA L4 (24 GB) or A100 (40 GB)

In [ ]:
# Quick sanity smoke: 20 anchors, fleet=64, K=8, H=100.
# Should finish in ~30 seconds; just confirms the script runs end-to-end on this Colab session.
%cd /content
%env NUMBA_THREADING_LAYER=omp
!python -m alphatrain.scripts.phase1_oracle_fleet_gpu \
    --model alphatrain/data/pillar2z_epoch_19.pt \
    --crisis-dir data/crisis_v12 \
    --selfplay-dir data/selfplay_v12 \
    --num-anchors 20 --crisis-frac 1.0 --selfplay-frac 0.0 \
    --top-moves 6 --k-continuations 8 --horizon 100 \
    --fleet-size 64 \
    --compile \
    --device cuda \
    --output /tmp/fleet_smoke.pt

## Production run

Adjust `NUM_ANCHORS`, `CRISIS_FRAC`, `K_CONT`, `H` as needed. Defaults below match the Path B requirement.

**Tuning `--fleet-size`:**
- L4 (24 GB): try 512 first, push to 1024 if GPU underutilized.
- A100 (40 GB): start at 1024.
- If OOM: drop to 256.

In [ ]:
# ── KNOBS ─────────────────────────────────────────────────────────────
NUM_ANCHORS  = 10000   # 10K anchors (crisis-heavy) for Path B
CRISIS_FRAC  = 1.0     # Crisis is where oracle margin is biggest
SELFPLAY_FRAC = 0.0    # Set to 0.5 / 0.5 if you want healthy-state coverage too
K_CONT       = 32      # K=32 common-RNG continuations per move
TOP_MOVES    = 6       # top-K policy moves judged per anchor
HORIZON      = 300     # rollout horizon in turns
FLEET_SIZE   = 1024     # concurrent rollouts (== GPU batch size)
SAMPLE_SEED  = 2027    # change for disjoint shards
OUT_NAME     = f'phase1_oracle_fleet_{NUM_ANCHORS}.pt'
# ──────────────────────────────────────────────────────────────────────

print(f'Anchors: {NUM_ANCHORS} (crisis_frac={CRISIS_FRAC}, selfplay_frac={SELFPLAY_FRAC})')
print(f'K={K_CONT}, top-{TOP_MOVES}, H={HORIZON}, fleet={FLEET_SIZE}')
print(f'Output: /content/{OUT_NAME}')
print(f'Will copy to: {DRIVE}/{OUT_NAME}')

In [ ]:
%env NUMBA_THREADING_LAYER=omp
# Production run. ETA on L4: ~2-4 hours for 10K crisis anchors at K=32/top-6/H=300.
import time
t0 = time.time()

!python -u -m alphatrain.scripts.phase1_oracle_fleet_gpu \
    --model alphatrain/data/pillar2z_epoch_19.pt \
    --crisis-dir data/crisis_v12 \
    --selfplay-dir data/selfplay_v12 \
    --num-anchors {NUM_ANCHORS} \
    --crisis-frac {CRISIS_FRAC} --selfplay-frac {SELFPLAY_FRAC} \
    --top-moves {TOP_MOVES} --k-continuations {K_CONT} --horizon {HORIZON} \
    --fleet-size {FLEET_SIZE} \
    --compile \
    --device cuda \
    --sample-seed {SAMPLE_SEED} \
    --output /content/{OUT_NAME}

print(f'\nTotal: {(time.time()-t0)/60:.1f} min')

In [ ]:
# Copy result back to Drive
!cp /content/{OUT_NAME} {DRIVE}/{OUT_NAME}
!ls -lh {DRIVE}/{OUT_NAME}

## After the mining completes

1. The file `phase1_oracle_fleet_10000.pt` is in your Drive.
2. Download it locally (`gdrive cp` or web UI).
3. Inspect quick audit numbers (P(oracle ≠ policy_top_1), Δcap_rate).
4. Use it as input to Path B training (mix with V12 self-play tensors).

## Sharding multiple Colab sessions

To run multiple Colab instances in parallel, change `SAMPLE_SEED` and `OUT_NAME` per session. With different seeds the anchor pools are disjoint random samples from the same source dirs.